# Notebook 6: Regularization — Dropout, Batch Normalization & Weight Decay

---

## Learning Objectives

By the end of this notebook you will be able to:

- Identify overfitting and underfitting from training/validation curves
- Apply L2 regularization (weight decay) and explain its effect
- Implement and use dropout correctly in training vs evaluation mode
- Explain what batch normalization does and why it helps training
- Use early stopping to prevent overfitting
- Compare the effectiveness of different regularization techniques

## Prerequisites

- Notebook 05 (Optimization)
- Understanding of training loops and loss computation
- Basic PyTorch (`nn.Module`, `DataLoader`)

## Table of Contents

1. [Overfitting vs Underfitting](#1-overfitting-vs-underfitting)
2. [L2 Regularization (Weight Decay)](#2-l2-regularization-weight-decay)
3. [Dropout](#3-dropout)
4. [Batch Normalization](#4-batch-normalization)
5. [Early Stopping](#5-early-stopping)
6. [Code: Demonstrate Overfitting and Fix It](#6-code-demonstrate-overfitting-and-fix-it)
7. [Code: Dropout Effect (Train vs Eval)](#7-code-dropout-effect-train-vs-eval)
8. [Code: Batch Norm Effect on Training Speed](#8-code-batch-norm-effect-on-training-speed)
9. [Exercise: Compare Regularization Techniques](#9-exercise)
10. [Common Mistakes & Debugging Tips](#10-common-mistakes--debugging-tips)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

np.random.seed(42)
torch.manual_seed(42)

%matplotlib inline
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['figure.dpi'] = 100

---
## 1. Overfitting vs Underfitting

**Overfitting:** The model memorizes training data (including noise) and performs poorly on unseen data.
- Training loss keeps decreasing, but validation loss starts increasing.

**Underfitting:** The model is too simple to capture the underlying pattern.
- Both training and validation losses are high.

**The goal:** Find the sweet spot where the model captures the true pattern without fitting noise.

In [ ]:
# Illustrate overfitting vs underfitting with synthetic curves
epochs = np.arange(1, 101)

# Simulated loss curves
train_loss = 1.0 * np.exp(-0.05 * epochs) + 0.02
val_loss_good = 1.0 * np.exp(-0.04 * epochs) + 0.08  # good fit
val_loss_overfit = 0.5 * np.exp(-0.08 * epochs) + 0.05 + 0.003 * epochs  # overfitting
train_loss_underfit = np.ones_like(epochs) * 0.6 + 0.05 * np.exp(-0.02 * epochs)
val_loss_underfit = np.ones_like(epochs) * 0.7 + 0.05 * np.exp(-0.02 * epochs)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Underfitting
axes[0].plot(epochs, train_loss_underfit, 'b-', label='Train', linewidth=2)
axes[0].plot(epochs, val_loss_underfit, 'r--', label='Validation', linewidth=2)
axes[0].set_title('Underfitting', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].set_ylim(0, 1.2)
axes[0].grid(True, alpha=0.3)

# Good fit
axes[1].plot(epochs, train_loss, 'b-', label='Train', linewidth=2)
axes[1].plot(epochs, val_loss_good, 'r--', label='Validation', linewidth=2)
axes[1].set_title('Good Fit', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].set_ylim(0, 1.2)
axes[1].grid(True, alpha=0.3)

# Overfitting
axes[2].plot(epochs, train_loss, 'b-', label='Train', linewidth=2)
axes[2].plot(epochs, val_loss_overfit, 'r--', label='Validation', linewidth=2)
axes[2].axvline(x=30, color='green', linestyle=':', alpha=0.7, label='Best epoch')
axes[2].set_title('Overfitting', fontsize=13)
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].set_ylim(0, 1.2)
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 2. L2 Regularization (Weight Decay)

L2 regularization adds a penalty proportional to the squared magnitude of the weights:

$$L_{\text{reg}} = L + \lambda \sum_{i} w_i^2$$

where $\lambda$ is the regularization strength.

**Effect on the gradient:**

$$\frac{\partial L_{\text{reg}}}{\partial w} = \frac{\partial L}{\partial w} + 2\lambda w$$

This means each update "decays" the weight toward zero:

$$w \leftarrow w - \eta\left(\frac{\partial L}{\partial w} + 2\lambda w\right) = (1 - 2\eta\lambda)\,w - \eta\frac{\partial L}{\partial w}$$

**Why it helps:**
- Prevents any single weight from becoming too large.
- Encourages the model to use all features rather than relying on a few.
- Smooths the learned function, reducing overfitting.

**In PyTorch:** Use the `weight_decay` parameter in the optimizer:
```python
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
```

---
## 3. Dropout

Dropout randomly sets a fraction $p$ of neuron activations to zero during training.

**During training:**
- Each neuron is "dropped" (output set to 0) with probability $p$.
- Surviving activations are scaled by $\frac{1}{1-p}$ to maintain expected values.

**During evaluation:**
- All neurons are active (no dropout).
- No scaling needed (already handled during training via inverted dropout).

**Why it works:**
- Prevents co-adaptation of neurons — no neuron can rely on another always being present.
- Acts like an ensemble of many sub-networks.
- Common values: $p = 0.2$ to $p = 0.5$.

**Critical:** You must call `model.train()` before training and `model.eval()` before inference!

---
## 4. Batch Normalization

Batch normalization normalizes each layer's inputs across the mini-batch:

$$\hat{x} = \frac{x - \mu_{\mathcal{B}}}{\sqrt{\sigma_{\mathcal{B}}^2 + \epsilon}}$$

followed by a learnable scale and shift:

$$y = \gamma \hat{x} + \beta$$

where:
- $\mu_{\mathcal{B}}$ = mini-batch mean
- $\sigma_{\mathcal{B}}^2$ = mini-batch variance
- $\gamma, \beta$ = learnable parameters
- $\epsilon$ = small constant for numerical stability

**Why it helps:**
- Reduces internal covariate shift (layer inputs stay well-distributed).
- Allows higher learning rates.
- Acts as a mild regularizer (due to mini-batch noise).
- Speeds up convergence significantly.

**In PyTorch:**
```python
nn.BatchNorm1d(num_features)  # after linear layers
nn.BatchNorm2d(num_channels)  # after conv layers
```

**Note:** During evaluation, batch norm uses running statistics (stored during training), not the current batch statistics.

---
## 5. Early Stopping

Early stopping monitors the validation loss and stops training when it stops improving.

**Algorithm:**
1. After each epoch, compute validation loss.
2. If val loss improves, save the model ("checkpoint") and reset the patience counter.
3. If val loss does not improve for `patience` consecutive epochs, stop training.
4. Restore the best model weights.

**Hyperparameters:**
- `patience`: How many epochs to wait (typical: 5–20).
- `min_delta`: Minimum improvement to count as "better" (typical: 0 or 1e-4).

Early stopping is the simplest and most effective regularization technique in practice.

---
## 6. Code: Demonstrate Overfitting and Fix It

We create a small synthetic dataset, train an overly large model, and then apply regularization techniques to reduce overfitting.

In [ ]:
# Create a small synthetic dataset (easy to overfit)
np.random.seed(42)
torch.manual_seed(42)

N_train, N_val = 50, 200
X_train_np = np.random.randn(N_train, 10).astype(np.float32)
y_train_np = (X_train_np[:, 0] + 0.5 * X_train_np[:, 1] - 0.3 * X_train_np[:, 2]
              + 0.3 * np.random.randn(N_train)).astype(np.float32)

X_val_np = np.random.randn(N_val, 10).astype(np.float32)
y_val_np = (X_val_np[:, 0] + 0.5 * X_val_np[:, 1] - 0.3 * X_val_np[:, 2]
            + 0.3 * np.random.randn(N_val)).astype(np.float32)

X_train = torch.from_numpy(X_train_np)
y_train = torch.from_numpy(y_train_np)
X_val = torch.from_numpy(X_val_np)
y_val = torch.from_numpy(y_val_np)

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)

In [ ]:
def train_model(model, optimizer, epochs=200):
    """Train model and return train/val loss histories."""
    loss_fn = nn.MSELoss()
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        # Training
        model.train()
        epoch_loss = 0.0
        n_batches = 0
        for xb, yb in train_loader:
            pred = model(xb).squeeze()
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n_batches += 1
        train_losses.append(epoch_loss / n_batches)
        
        # Validation
        model.eval()
        with torch.no_grad():
            val_pred = model(X_val).squeeze()
            val_loss = loss_fn(val_pred, y_val).item()
        val_losses.append(val_loss)
    
    return train_losses, val_losses

# --- No regularization (overfit) ---
torch.manual_seed(42)
model_overfit = nn.Sequential(
    nn.Linear(10, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 1)
)
opt_overfit = optim.Adam(model_overfit.parameters(), lr=0.001)
train_l_of, val_l_of = train_model(model_overfit, opt_overfit)

# --- With weight decay ---
torch.manual_seed(42)
model_wd = nn.Sequential(
    nn.Linear(10, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 1)
)
opt_wd = optim.Adam(model_wd.parameters(), lr=0.001, weight_decay=1e-2)
train_l_wd, val_l_wd = train_model(model_wd, opt_wd)

# --- With dropout ---
torch.manual_seed(42)
model_drop = nn.Sequential(
    nn.Linear(10, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 128), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 1)
)
opt_drop = optim.Adam(model_drop.parameters(), lr=0.001)
train_l_dr, val_l_dr = train_model(model_drop, opt_drop)

print(f"Final val loss (no reg):      {val_l_of[-1]:.4f}")
print(f"Final val loss (weight decay): {val_l_wd[-1]:.4f}")
print(f"Final val loss (dropout):      {val_l_dr[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

for ax, title, tl, vl in [
    (axes[0], 'No Regularization', train_l_of, val_l_of),
    (axes[1], 'Weight Decay (1e-2)', train_l_wd, val_l_wd),
    (axes[2], 'Dropout (p=0.3)', train_l_dr, val_l_dr),
]:
    ax.plot(tl, 'b-', label='Train', alpha=0.8)
    ax.plot(vl, 'r--', label='Validation', alpha=0.8)
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('Epoch')
    ax.legend()
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel('MSE Loss')
plt.suptitle('Effect of Regularization on Overfitting', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Code: Dropout Effect (Train vs Eval)

A critical aspect of dropout: it behaves differently in training and evaluation modes. Forgetting to switch to `model.eval()` during inference is a common bug.

In [ ]:
torch.manual_seed(42)

# Simple model with dropout
model_demo = nn.Sequential(
    nn.Linear(5, 20), nn.ReLU(), nn.Dropout(0.5),
    nn.Linear(20, 1)
)

x_test = torch.randn(1, 5)

# --- Training mode: outputs are stochastic ---
model_demo.train()
print("=== Training mode (dropout ACTIVE) ===")
print("Multiple forward passes give DIFFERENT outputs:")
for i in range(5):
    out = model_demo(x_test)
    print(f"  Pass {i+1}: {out.item():.4f}")

# --- Eval mode: outputs are deterministic ---
model_demo.eval()
print("\n=== Eval mode (dropout INACTIVE) ===")
print("Multiple forward passes give the SAME output:")
for i in range(5):
    out = model_demo(x_test)
    print(f"  Pass {i+1}: {out.item():.4f}")

In [ ]:
# Visualize the effect of dropout on prediction variance
torch.manual_seed(42)
x_range = torch.linspace(-3, 3, 100).unsqueeze(1)

# Small model trained on sine data
model_sine = nn.Sequential(
    nn.Linear(1, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 64), nn.ReLU(), nn.Dropout(0.3),
    nn.Linear(64, 1)
)

# Quick training
X_sine = torch.linspace(-3, 3, 100).unsqueeze(1)
y_sine = torch.sin(X_sine)
opt_sine = optim.Adam(model_sine.parameters(), lr=0.01)
for _ in range(500):
    model_sine.train()
    pred = model_sine(X_sine)
    loss = nn.MSELoss()(pred, y_sine)
    opt_sine.zero_grad()
    loss.backward()
    opt_sine.step()

# Predictions in train mode (stochastic) vs eval mode (deterministic)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Train mode: multiple noisy predictions
model_sine.train()
for i in range(20):
    with torch.no_grad():
        pred = model_sine(x_range)
    axes[0].plot(x_range.numpy(), pred.numpy(), 'b-', alpha=0.15)
axes[0].plot(x_range.numpy(), torch.sin(x_range).numpy(), 'r-', linewidth=2, label='True')
axes[0].set_title('Train Mode (dropout ON): stochastic predictions')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Eval mode: single deterministic prediction
model_sine.eval()
with torch.no_grad():
    pred_eval = model_sine(x_range)
axes[1].plot(x_range.numpy(), pred_eval.numpy(), 'b-', linewidth=2, label='Prediction')
axes[1].plot(x_range.numpy(), torch.sin(x_range).numpy(), 'r--', linewidth=2, label='True')
axes[1].set_title('Eval Mode (dropout OFF): deterministic prediction')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 8. Code: Batch Norm Effect on Training Speed

We compare training with and without batch normalization on a deeper network.

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

# Generate a larger dataset for this comparison
N = 500
X_data = torch.randn(N, 20)
# Target: nonlinear function of first few features
y_data = (torch.sin(X_data[:, 0]) + X_data[:, 1]**2 - X_data[:, 2]
          + 0.2 * torch.randn(N))

train_loader_bn = DataLoader(TensorDataset(X_data, y_data), batch_size=32, shuffle=True)

# --- Model WITHOUT batch norm ---
torch.manual_seed(42)
model_no_bn = nn.Sequential(
    nn.Linear(20, 128), nn.ReLU(),
    nn.Linear(128, 128), nn.ReLU(),
    nn.Linear(128, 64), nn.ReLU(),
    nn.Linear(64, 32), nn.ReLU(),
    nn.Linear(32, 1)
)

# --- Model WITH batch norm ---
torch.manual_seed(42)
model_with_bn = nn.Sequential(
    nn.Linear(20, 128), nn.BatchNorm1d(128), nn.ReLU(),
    nn.Linear(128, 128), nn.BatchNorm1d(128), nn.ReLU(),
    nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(),
    nn.Linear(64, 32), nn.BatchNorm1d(32), nn.ReLU(),
    nn.Linear(32, 1)
)

def train_epoch_losses(model, loader, epochs=100, lr=0.001):
    """Train and return per-epoch losses."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []
    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n = 0
        for xb, yb in loader:
            pred = model(xb).squeeze()
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n += 1
        losses.append(epoch_loss / n)
    return losses

losses_no_bn = train_epoch_losses(model_no_bn, train_loader_bn, epochs=150)
losses_with_bn = train_epoch_losses(model_with_bn, train_loader_bn, epochs=150)

plt.figure(figsize=(8, 5))
plt.plot(losses_no_bn, 'b-', label='Without BatchNorm', alpha=0.8)
plt.plot(losses_with_bn, 'r-', label='With BatchNorm', alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Speed: With vs Without Batch Normalization')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final loss without BN: {losses_no_bn[-1]:.4f}")
print(f"Final loss with BN:    {losses_with_bn[-1]:.4f}")

**Observation:** Batch normalization typically helps the network converge faster and to a lower loss, especially for deeper networks.

---
<a id='9-exercise'></a>
## 9. Exercise: Compare Regularization Techniques on a Synthetic Dataset

**Task:** Use the synthetic dataset below and compare the validation loss of:
1. No regularization (baseline)
2. Weight decay = 0.01
3. Dropout = 0.4
4. Batch normalization
5. Dropout + batch normalization + weight decay (combined)

Plot all validation curves on the same figure and determine which technique (or combination) works best.

In [ ]:
# ============================================================
# EXERCISE: Complete the code below
# ============================================================

# torch.manual_seed(42)
# np.random.seed(42)
#
# # Dataset: 80 train, 400 val, 15 features, only 3 are relevant
# N_tr, N_vl, D = 80, 400, 15
# X_tr = torch.randn(N_tr, D)
# y_tr = X_tr[:, 0] - 2 * X_tr[:, 1] + 0.5 * X_tr[:, 2] + 0.3 * torch.randn(N_tr)
# X_vl = torch.randn(N_vl, D)
# y_vl = X_vl[:, 0] - 2 * X_vl[:, 1] + 0.5 * X_vl[:, 2] + 0.3 * torch.randn(N_vl)
#
# loader_tr = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)
#
# def build_model(use_dropout=False, use_batchnorm=False):
#     layers = []
#     # TODO: Build a 3-hidden-layer MLP (D -> 128 -> 64 -> 32 -> 1)
#     # Optionally add nn.Dropout(0.4) and/or nn.BatchNorm1d
#     pass
#     return nn.Sequential(*layers)
#
# # TODO: Train each variant and collect val losses
# # TODO: Plot all val loss curves on one figure
# # Hint: reuse the train_model function pattern from Section 6

### Solution

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

N_tr, N_vl, D = 80, 400, 15
X_tr = torch.randn(N_tr, D)
y_tr = X_tr[:, 0] - 2 * X_tr[:, 1] + 0.5 * X_tr[:, 2] + 0.3 * torch.randn(N_tr)
X_vl = torch.randn(N_vl, D)
y_vl = X_vl[:, 0] - 2 * X_vl[:, 1] + 0.5 * X_vl[:, 2] + 0.3 * torch.randn(N_vl)

loader_tr = DataLoader(TensorDataset(X_tr, y_tr), batch_size=16, shuffle=True)

def build_model(use_dropout=False, use_batchnorm=False, p=0.4):
    """Build a 3-hidden-layer MLP with optional regularization."""
    layers = []
    in_features = [D, 128, 64, 32]
    out_features = [128, 64, 32]
    
    for i, (inf, outf) in enumerate(zip(in_features[:-1], out_features)):
        layers.append(nn.Linear(inf, outf))
        if use_batchnorm:
            layers.append(nn.BatchNorm1d(outf))
        layers.append(nn.ReLU())
        if use_dropout:
            layers.append(nn.Dropout(p))
    layers.append(nn.Linear(32, 1))
    return nn.Sequential(*layers)

def train_and_eval(model, weight_decay=0.0, epochs=200):
    """Train with given weight_decay and return train/val losses."""
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()
    train_losses, val_losses = [], []
    
    for epoch in range(epochs):
        model.train()
        epoch_loss, n = 0.0, 0
        for xb, yb in loader_tr:
            pred = model(xb).squeeze()
            loss = loss_fn(pred, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()
            n += 1
        train_losses.append(epoch_loss / n)
        
        model.eval()
        with torch.no_grad():
            val_losses.append(loss_fn(model(X_vl).squeeze(), y_vl).item())
    
    return train_losses, val_losses

configs = {
    'No reg': (False, False, 0.0),
    'Weight decay': (False, False, 0.01),
    'Dropout': (True, False, 0.0),
    'BatchNorm': (False, True, 0.0),
    'All combined': (True, True, 0.01),
}

results = {}
for name, (do, bn, wd) in configs.items():
    torch.manual_seed(42)
    model = build_model(use_dropout=do, use_batchnorm=bn)
    tl, vl = train_and_eval(model, weight_decay=wd)
    results[name] = (tl, vl)
    print(f"{name:15s} -> final val loss: {vl[-1]:.4f}, best val loss: {min(vl):.4f}")

plt.figure(figsize=(10, 5))
for name, (tl, vl) in results.items():
    plt.plot(vl, label=name, alpha=0.8)
plt.xlabel('Epoch')
plt.ylabel('Validation MSE')
plt.title('Regularization Technique Comparison (Validation Loss)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

---
## 10. Common Mistakes & Debugging Tips

| Mistake | Symptom | Fix |
|---|---|---|
| Forgetting `model.eval()` during inference | Predictions are noisy/wrong (dropout still active) | Always call `model.eval()` before testing |
| Forgetting `model.train()` after evaluation | Model stops learning (dropout/BN frozen) | Call `model.train()` at the start of each epoch |
| Too much dropout ($p > 0.5$) | Underfitting; loss barely decreases | Reduce dropout rate; try $p=0.1$–$0.3$ |
| Weight decay too large | Weights collapse to near-zero; underfitting | Reduce by 10x; typical range: $10^{-5}$ to $10^{-2}$ |
| BatchNorm with batch_size=1 | Crashes or NaN (cannot compute variance of 1 sample) | Use batch_size >= 2; consider LayerNorm for small batches |
| Applying L2 regularization to bias terms | Usually unnecessary; can hurt performance | PyTorch's `weight_decay` applies to all params by default; use parameter groups to exclude biases |
| Not using early stopping | Wasted compute and potential overfitting | Always monitor val loss; use early stopping |

**Debugging checklist:**
1. Always plot train AND validation loss — the gap reveals overfitting.
2. Start without regularization, confirm the model can overfit, then add regularization.
3. Apply one technique at a time to understand each effect.
4. For very small datasets, weight decay and dropout are usually most effective.
5. For deep networks, batch normalization is almost always beneficial.